In [77]:
from dotenv import load_dotenv

load_dotenv()

True

# Wedding Planner Agent

- **Main Wedding Planner Agent:** Asks about the wedding plan. 

    For e.g: Place of wedding, Venue, activities, food, etc.

- **Flight Booker Agent:** Books flights for the wedding place/country/city.

    For e.g: Karachi to Paris

- **Venue Finder Agent:** Finds the best venue in/around the decided wedding place.

    For e.g: If wedding is in Paris, it looks for the best wedding hall/venue in Paris.

- **Connoisseur Agent:** Plans the food items to have in the wedding

    For e.g: What foods to put, drinks, desserts, etc.

In [78]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

In [79]:
from langchain.agents import create_agent

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient


client = MultiServerMCPClient(
   {
    "kiwi-mcp-server": {
      "url": "https://mcp-servers--kiwi-mcp-server.apify.actor/mcp",
      "headers": {
        "Authorization": "Bearer YOUR_APIFY_TOKEN"
      }
    }
  } 
)

In [25]:
from pprint import pprint

tools = await client.get_tools()

pprint(tools)

Encountered SSE exception
Traceback (most recent call last):
  File "c:\Agentic-AI-Engineering\PHASE_02\langchain\lca-lc-foundations\.venv\Lib\site-packages\mcp\client\sse.py", line 81, in sse_reader
    async for sse in event_source.aiter_sse():  # pragma: no branch
  File "c:\Agentic-AI-Engineering\PHASE_02\langchain\lca-lc-foundations\.venv\Lib\site-packages\httpx_sse\_api.py", line 38, in aiter_sse
    self._check_content_type()
  File "c:\Agentic-AI-Engineering\PHASE_02\langchain\lca-lc-foundations\.venv\Lib\site-packages\httpx_sse\_api.py", line 19, in _check_content_type
    raise SSEError(
httpx_sse._exceptions.SSEError: Expected response header Content-Type to contain 'text/event-stream', got 'text/html'
  + Exception Group Traceback (most recent call last):
  |   File "c:\Agentic-AI-Engineering\PHASE_02\langchain\lca-lc-foundations\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3745, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)


In [80]:
from langchain.tools import tool

@tool
def initiate_flight_booking(from_city: str, to_city: str):
    """Only initiate flights booking. Flights will be PENDING until user confirms"""
    return f"Flight from {from_city} to {to_city} pending to be booked."

@tool
def confirm_flight_booking(from_city: str, to_city: str):
    """Confirm flight booking. Flight Booking status should be CONFIRMED"""
    return f"Flight booked from {from_city} to {to_city}."


In [81]:
fb_system_prompt = """
You are a flight booker agent. Your job is to book flights from one place to another.
Once you know about the departure city (from_city), and destination city (to_city), use your tools wisely to initiate the flight booking which will be pending at first.
Only  after you get confirmation from the user, only then confirm the flight booking using your tools
"""

flight_booker_agent =  create_agent(
    model="gpt-5-nano",
    system_prompt=fb_system_prompt,
    tools=[initiate_flight_booking, confirm_flight_booking]
)

In [83]:
from langchain.tools import tool
from tavily.client import TavilyClient

@tool
def find_venue(city: str):
    """Find the best wedding hall/venue in the city the user wants to go for their wedding.
    You can get the destination information about the city from the flight_booking_agent output"""

    tavily_client = TavilyClient()

    search_result = tavily_client.search(f"Best Wedding Halls/Venues in {city}")

    return search_result

@tool
def confirm_venue_booking(venue: str):
    """Confirm venue booking"""

    return f"Congratulations! {venue} is booked for your wedding!"

In [84]:
vf_system_prompt = """
You are a vanue agent. 
Your job is to find the best wedding venue in the told city/place based on user preferences using your tools.
Once you tell the user about some of the best wedding venues, ask them to confirm it.
Once user confirms the venue, only then confirm the venue booking using your tools.
"""

venue_finder_agent =  create_agent(
    model="gpt-5-nano",
    system_prompt=vf_system_prompt,
    tools=[find_venue, confirm_venue_booking]
)

In [85]:
c_system_prompt = """
You are a connoisseur agent. 
You are an expert at designing menus for weddings.
Your job is to plan the best wedding menu based on user preferences.
"""

connoisseur_agent =  create_agent(
    model="gpt-5-nano",
    system_prompt=c_system_prompt
)

In [86]:
from langchain.messages import HumanMessage
from langchain.tools import tool

@tool
def book_flights(from_city: str, to_city: str):
    """Book flights for the user for their wedding by having the from_city and to_city"""
    response = flight_booker_agent.invoke({"messages":[HumanMessage(content=f"Book flights from {from_city} to {to_city}")]})
    return response["messages"][-1].content

@tool
def arrange_venue(city: str):
    """Arrange the best wedding hall/venue for the user in the city they're going to"""
    response = venue_finder_agent.invoke({"messages":[HumanMessage(content=f"Arrange venue in {city}")]})
    return response["messages"][-1].content

@tool
def design_menu():
    """design the best wedding food menu for the users wedding based on their preferences"""
    response = flight_booker_agent.invoke({"messages":[HumanMessage(content="Design a great wedding food menu based on user preferences.")]})
    return response["messages"][-1].content

In [91]:
from langchain.agents import create_agent

system_prompt = """
You are the Main Wedding Planner Agent. 
Your job is to lead the user through their wedding planning journey by gathering their core vision and delegating tasks to specialist agents when needed. 

Keep responses very brief and short.

First understand the query properly.

Dont ask unnecessary details.

You only need:
for booking flights, you need the departure city, destination city, use your book_flights tool 
for arranging venue, you need the city the user is flying tool, use your arrange_venue tool
for designing menu, you need user preferences, use your design_menu tool

use your tools when necessary

Keep responses very brief and short.

"""

wedding_planner_agent = create_agent(
    model="gpt-5-nano",
    system_prompt=system_prompt,
    tools=[book_flights, arrange_venue, design_menu],
    checkpointer=memory
)

In [92]:
from langchain.messages import HumanMessage

config = {"configurable":{"thread_id":1}}

query = "heyy im going to marry. I love paris. Can you book my flights from karachi to paris and find a great wedding hall there as well. also I want you create a food menu for my wedding as well"
response = wedding_planner_agent.invoke({"messages":[HumanMessage(content=query)]}, config=config)
response

{'messages': [HumanMessage(content='heyy im going to marry. I love paris. Can you book my flights from karachi to paris and find a great wedding hall there as well. also I want you create a food menu for my wedding as well', additional_kwargs={}, response_metadata={}, id='a015662e-9b24-424b-a8cf-36d2c1438ac3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 779, 'prompt_tokens': 392, 'total_tokens': 1171, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EALXln29S2M2q8UaiiV34Io520wMu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fddee-2abe-7e03-a8a5-91c09c9ad443-0', tool_calls=[{'name': 'book_flights', 'ar

In [93]:
from pprint import pprint

pprint(response["messages"][-1].content)

('Got it. Brief update:\n'
 '\n'
 '- Flights: booking is pending. Please confirm:\n'
 '  - One-way or round-trip? depart date (and return date if round-trip)\n'
 '  - Number of passengers\n'
 '  - Cabin class (Economy, Premium, Business, First)\n'
 '  - Airline preferences or cheapest option?\n'
 '  - Any special requests (seats, meals, assistance)\n'
 '\n'
 '- Venues in Paris (top options):\n'
 '  - Shangri-La Paris, Ritz Paris, Hôtel Le Meurice, The Peninsula Paris, '
 'Hôtel de Crillon\n'
 '  - Want me to check availability and hold one? If yes, share:\n'
 '    - Guest count, target date, indoor/outdoor, budget, must-haves\n'
 '\n'
 '- Menu design: ready to plan. Please share:\n'
 '  - Cuisine style, number of courses\n'
 '  - Dietary needs (vegetarian, vegan, halal, allergies)\n'
 '  - Guest count, service style (plated, buffet), beverage preferences\n'
 '\n'
 'Send details or pick a venue to proceed.')


In [94]:
query = """Round-trip, depart June 10 and return June 15. 50 passengers, Economy, cheapest option. No special requests.

Venue: 50 guests, June 12, indoor, $20,000 budget, elegant setting.

Menu: Pakistani/Italian fusion, 4 courses, halal, plated service, soft drinks and mocktails."""
response = wedding_planner_agent.invoke({"messages":[HumanMessage(content=query)]}, config=config)
response

{'messages': [HumanMessage(content='heyy im going to marry. I love paris. Can you book my flights from karachi to paris and find a great wedding hall there as well. also I want you create a food menu for my wedding as well', additional_kwargs={}, response_metadata={}, id='a015662e-9b24-424b-a8cf-36d2c1438ac3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 779, 'prompt_tokens': 392, 'total_tokens': 1171, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EALXln29S2M2q8UaiiV34Io520wMu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fddee-2abe-7e03-a8a5-91c09c9ad443-0', tool_calls=[{'name': 'book_flights', 'ar

In [95]:
from pprint import pprint

pprint(response["messages"][-1].content)

("All set. Here's the status:\n"
 '\n'
 '- Flights: Pending. Details — Karachi to Paris, round-trip June 10–June 15, '
 '50 passengers, Economy, cheapest option, no special requests. To finalize, '
 'reply “confirm” (or tweak any detail).\n'
 '\n'
 '- Venue: Paris, 50 guests, June 12, indoor, budget $20,000, elegant. I can '
 'check availability and hold a top option. Want me to proceed with checks and '
 'place a hold on a preferred venue?\n'
 '\n'
 '- Menu: Pakistani/Italian fusion, 4 courses, halal, plated service, soft '
 'drinks and mocktails. Menu draft in progress. Ready to finalize a draft—any '
 'tweaks before I lock it in?')


In [96]:

query = "yes pls confirm everything"
response = wedding_planner_agent.invoke({"messages":[HumanMessage(content=query)]}, config=config)
response

{'messages': [HumanMessage(content='heyy im going to marry. I love paris. Can you book my flights from karachi to paris and find a great wedding hall there as well. also I want you create a food menu for my wedding as well', additional_kwargs={}, response_metadata={}, id='a015662e-9b24-424b-a8cf-36d2c1438ac3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 779, 'prompt_tokens': 392, 'total_tokens': 1171, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EALXln29S2M2q8UaiiV34Io520wMu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fddee-2abe-7e03-a8a5-91c09c9ad443-0', tool_calls=[{'name': 'book_flights', 'ar

In [97]:
from pprint import pprint

pprint(response["messages"][-1].content)

('Thanks. Here’s the plan and status:\n'
 '\n'
 'Flights\n'
 '- Type: Round-trip\n'
 '- Route: Karachi (KHI) → Paris (CDG/ORY)\n'
 '- Dates: June 10–June 15\n'
 '- Passengers: 50\n'
 '- Class: Economy, cheapest option\n'
 '- Status: Pending\n'
 '- Needed to finalize: passenger details (full names as on passport, DOB, '
 'nationalities), dates confirmed, preferred airports (CDG or ORY), contact '
 'email/phone, any seating or meal requests.\n'
 '\n'
 'Venues in Paris\n'
 '- Shortlist options: Ritz Paris, Shangri-La Paris, Le Meurice, Hôtel de '
 'Crillon, Peninsula Paris\n'
 '- Status: Ready to check availability and hold a venue\n'
 '- Details you gave: 50 guests, June 12, indoor, budget $20,000, elegant\n'
 '- Next: pick 1–2 venues to check, share any must-haves (e.g., specific '
 'room), and confirm date.\n'
 '\n'
 'Menu\n'
 '- Style: Pakistani/Italian fusion\n'
 '- Courses: 4\n'
 '- Service: Plated\n'
 '- Dietary: Halal\n'
 '- Beverages: Soft drinks and mocktails\n'
 '- Status: Draf